In [1]:
import pickle
import numpy as np
import os

file = open('inputs/wilson/random_weekeday_2.pkl', 'rb')
payload_wilson_initial = pickle.load(file)
file.close()

In [2]:
from rtv_solver import OnlineRTVSolver

# Initialize the RTV solver with the URL of the OSRM server
online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:50000/")

In [3]:
# creating a new payload with new requests
# consider all requests that start before 05:40:00

current_time = 5*3600+30*60
step = 10*60

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [4]:
# create a new payload with the selected requests
new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": payload_wilson_initial["driver_runs"],}

## Fast Heuristic method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_heuristic(new_payload)
unserved_requests

[]

In [5]:
# Simulate to 5:40:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [6]:
# creating a new payload with new requests
# consider all requests that start between 05:40:00 and 05:50:00

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

2

In [7]:
# create a new payload with the selected requests
new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": selected_requests,
    "driver_runs": simulated_driver_runs,}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

[]

In [8]:
# Simulate to 5:50:00

current_time += step
simulated_driver_runs = online_rtv_solver.simulate_manifest(current_time,new_driver_runs,intermediate_location=False)

In [9]:
# creating a new payload with new requests
# consider all requests that are between before 05:50:00 and after 05:60:00

selected_requests = []
for request in payload_wilson_initial["requests"]:
    if request["pickup_time_window_start"] >= current_time and request["pickup_time_window_start"] < current_time+step:
        selected_requests.append(request)

len(selected_requests)

3

In [10]:
req = selected_requests[0]

# check feasibility of time slots


new_payload = {
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'time_windows' : [
            {'pickup_time_window_start': req['pickup_time_window_start'], 'pickup_time_window_end': req['pickup_time_window_start'] + 60, 'dropoff_time_window_start': req['dropoff_time_window_start'], 'dropoff_time_window_end': req['dropoff_time_window_start'] + 180},
            {'pickup_time_window_start': req['pickup_time_window_start']+900, 'pickup_time_window_end': req['pickup_time_window_end']+900, 'dropoff_time_window_start': req['dropoff_time_window_start']+900, 'dropoff_time_window_end': req['dropoff_time_window_end']+900},
            {'pickup_time_window_start': req['pickup_time_window_start']+1800, 'pickup_time_window_end': req['pickup_time_window_end']+1800, 'dropoff_time_window_start': req['dropoff_time_window_start']+1800, 'dropoff_time_window_end': req['dropoff_time_window_end']+1800},
        ],
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}


feasible_windows = online_rtv_solver.check_feasibility(new_payload)
feasible_windows

[({'pickup_time_window_start': 21966,
   'pickup_time_window_end': 23766,
   'dropoff_time_window_start': 22661,
   'dropoff_time_window_end': 24461},
  1.1346762589928059),
 ({'pickup_time_window_start': 22866,
   'pickup_time_window_end': 24666,
   'dropoff_time_window_start': 23561,
   'dropoff_time_window_end': 25361},
  1.1513669064748202)]

In [11]:
feasible_windows[0] # this has the feasible time slot and associated VMT/PMT ratio for this slot

({'pickup_time_window_start': 21966,
  'pickup_time_window_end': 23766,
  'dropoff_time_window_start': 22661,
  'dropoff_time_window_end': 24461},
 1.1346762589928059)

In [12]:
# Creating a request with infeasible time slots

req = selected_requests[0]

new_payload = {
    "depot": payload_wilson_initial["depot"],
    "requests": [
    {
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'pickup_time_window_start': req['pickup_time_window_start'], 
        'pickup_time_window_end': req['pickup_time_window_start']+180, 
        'dropoff_time_window_start': req['dropoff_time_window_start'], 
        'dropoff_time_window_end': req['dropoff_time_window_start']+180,
        'am': req['am'],
        'wc': req['wc']
    }],
    "driver_runs": simulated_driver_runs
}

## Full RTV method
new_driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
unserved_requests

[6.0]

In [16]:
# Serve at earliest possible time

new_driver_runs = online_rtv_solver.serve_asap(new_payload)

In [17]:
# Reoptimize the driver runs

repotimized_driver_runs = online_rtv_solver.resolve_pdptw_rtv({"depot": payload_wilson_initial["depot"], "driver_runs": new_driver_runs})